# clef-compactor demo

End-to-end walkthrough of query-aware context compaction. The demo drives the real `ClefCompactor` pipeline against a canned Clef API (`httpx.MockTransport`), so every cell runs offline with no credentials. The last cell shows the two lines to change for a live run.

In [ ]:
import httpx

from clef_compactor import ClefCompactor
from clef_compactor.client import ClefClient
from clef_compactor.config import Settings

# A canned Clef endpoint: noul answers exactly like the hosted API returns.
ANSWERS = {
    "chunk_1": {"type": "noul", "noul": 0.93},
    "chunk_2": {"type": "noul", "noul": 0.05},
    "chunk_3": {"type": "noul", "noul": 0.85},
    "chunk_4": {"type": "noul", "noul": 0.42},
    "chunk_5": {"type": "noul", "noul": 0.71},
}

def canned_transport(request: httpx.Request) -> httpx.Response:
    return httpx.Response(200, json={
        "result": {
            "model": "@cf/cloudflare/clef",
            "answers": ANSWERS,
            "usage": {"input_tokens": 412, "output_tokens": 96},
        },
        "success": True, "errors": [], "messages": [],
    }, headers={"cf-ray": "demo-ray-001"})

settings = Settings(account_id="demo", api_token="demo", max_retries=0)
compactor = ClefCompactor(
    account_id="demo", api_token="demo",
    client=ClefClient(settings, transport=httpx.MockTransport(canned_transport)),
)
print("compactor ready (canned API)")

In [ ]:
query = "What is the refund policy?"
chunks = [
    "Our refund policy allows returns within 30 days of purchase.",
    "The Eiffel Tower is located in Paris, France.",
    "Refunds go back to the original payment method in 5 business days.",
    "Office hours are Monday to Friday, 9am to 5pm.",
    "Email support@example.com with your order number to start a refund.",
]

result = compactor.compact(query, chunks, token_budget=60)
print(f"query: {result.query!r}")
print(f"kept {len(result.kept)} / dropped {len(result.dropped)} of {len(chunks)} chunks")
result

In [ ]:
for chunk in result.kept:
    print(f"KEEP  idx={chunk.index}  P={chunk.score:.2f}  tokens={chunk.tokens:3d}  {chunk.text}")
print()
for chunk in result.dropped:
    print(f"CUT   idx={chunk.index}  P={chunk.score:.2f}  reason={chunk.drop_reason.value}")

## The compacted context

`kept_texts()` returns the surviving chunks verbatim, ranked best first. Join them and the string goes straight into your prompt.

In [ ]:
context = "\n\n".join(result.kept_texts())
print(context)

In [ ]:
print(f"input tokens:  {result.total_input_tokens}")
print(f"output tokens: {result.total_output_tokens}")
print(f"saved:         {result.saved_fraction:.1%}")
print(f"api cost:      ${result.cost_estimate:.6f} (input tokens at $0.24/M)")
print(f"api usage:     {result.usage}")
print(f"latency:       {result.latency_ms:.1f} ms")

## Going live

Two changes turn this into a real run against Cloudflare: export `CLEF_ACCOUNT_ID` and `CLEF_API_TOKEN`, then build the compactor without an injected client:

```python
compactor = ClefCompactor()                    # sync
compactor = AsyncClefCompactor(model="clef-flash")  # or async, faster model
```